# 日本語 BERT モデルによる文書分類

日本語 BERT モデルをファインチューニングして文書分類します。

下では GPU を用いて深層学習モデルのファインチューニングを行うので、「ランタイム」→「ランタイムのタイプを変更」からハードウェア アクセラレータを `GPU`（T4 など）にしておいてください（CPU のままだと大幅に時間がかかります）。

In [ ]:
import torch

if torch.cuda.is_available():
    print("使用可能な GPU:", torch.cuda.get_device_name(0))
else:
    print("警告: GPU が見つかりません（CPU で実行されます）。Colab の「ランタイム」→「ランタイムのタイプを変更」から GPU を選択してください。")

## 準備

必要なパッケージや辞書をインストールします。

In [ ]:
!pip install -q "transformers[ja]" sentencepiece datasets evaluate

In [ ]:
import pandas as pd
from transformers import BertJapaneseTokenizer, BertForSequenceClassification, pipeline

## 日本語 BERT

日本語 BERT の事前学習モデルは HuggingFace Transformers で公開されているものが一番使いやすいです。

今回は東北大の BERT-base モデル (v2) を使ってみます（以前は cl-tohoku という名前で公開されていましたが、現在は tohoku-nlp という名前に変わっています）。これは日本語 Wikipedia データセットに対して MeCab で UniDic を用いて形態素解析されたもので、さらに WordPiece アルゴリズムを用いてサブワード化が行われたものが公開されています。

- https://huggingface.co/tohoku-nlp
- https://github.com/cl-tohoku/bert-japanese

> **補足**: より新しい後継モデルとして [`tohoku-nlp/bert-base-japanese-v3`](https://huggingface.co/tohoku-nlp/bert-base-japanese-v3) も公開されています。また、`BertJapaneseTokenizer` / `BertForSequenceClassification` の代わりに `AutoTokenizer` / `AutoModelForSequenceClassification` を使うと、モデル名を変えるだけで別の事前学習モデルへ簡単に差し替えることができます。

## マスク言語モデルによる推論

BERT はマスク言語モデルによって事前学習されています。そのため、事前学習モデルを使えば、文中の任意のトークンをマスクしても予測を行うことができます。

In [ ]:
model_name = "tohoku-nlp/bert-base-japanese-v2"

tokenizer = BertJapaneseTokenizer.from_pretrained(model_name)

fitb = pipeline('fill-mask', model=model_name, tokenizer=tokenizer)
fitb("私は今朝[MASK]を食べました。")

## 文書分類のためのファインチューニング

2値分類タスクでファインチューニングを行います。
データは第5章で自ら作成したデータセットを使っていただく想定ですが、サンプルデータを用意しました。新型コロナウイルスに関する体験談風の文書1,000件に、「書き手（またはその家族）が新型コロナウイルスに感染したことが書かれているか」を TRUE/FALSE でラベル付けしたものです（TRUE と FALSE が500件ずつ）。「コメント」の列にラベルを付けた理由を書いてあります。

### データセット

In [ ]:
!curl -fLO https://raw.githubusercontent.com/mamorlis/nlpbook/refs/heads/main/misc/covid-annotation.csv

In [ ]:
df = pd.read_csv("covid-annotation.csv", usecols=["ラベル", "文書", "コメント"])
df = df.rename(columns={"ラベル": "label", "文書": "text", "コメント": "comment"})
df["label"] = df["label"].astype(int)
print(len(df))
print(df["label"].value_counts())
df.head()

In [ ]:
from datasets import Dataset, ClassLabel

dataset = Dataset.from_pandas(df).cast_column("label", ClassLabel(names=["FALSE", "TRUE"]))

# 全体の10%をテストデータ、残りの10%を開発データ、それ以外を訓練データにする
# stratify_by_column でラベルの比率を揃え、seed で毎回同じ分割になるようにする
# train_test_data["train"] => これを訓練データと開発データに分割する
# train_test_data["test"] => テストデータ
# train_dev_data["train"] => 訓練データ
# train_dev_data["test"]  => 開発データ
train_test_data = dataset.train_test_split(test_size=0.1, stratify_by_column="label", seed=42)
train_dev_data  = train_test_data["train"].train_test_split(test_size=0.1, stratify_by_column="label", seed=42)

def encode(examples):
    return tokenizer(examples['text'], truncation=True, max_length=512)

# text・comment 列は訓練には使わないので消しておく（warning を消すため）
train_dataset = train_dev_data["train"].map(encode, batched=True, remove_columns=["text", "comment"])
dev_dataset   = train_dev_data["test"].map(encode, batched=True, remove_columns=["text", "comment"])
test_dataset  = train_test_data["test"].map(encode, batched=True, remove_columns=["text", "comment"])
print(len(train_dataset), len(dev_dataset), len(test_dataset))

### モデル

In [ ]:
# id2label を指定しておくと、推論結果に LABEL_0 ではなく FALSE/TRUE と表示される
model = BertForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    id2label={0: "FALSE", 1: "TRUE"},
    label2id={"FALSE": 0, "TRUE": 1},
)

### 評価尺度

正解率（accuracy）に加えて、TRUE のクラスについての F1（適合率と再現率の調和平均）も計算します。

In [ ]:
import numpy as np
import evaluate

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {**accuracy.compute(predictions=predictions, references=labels),
            **f1.compute(predictions=predictions, references=labels)}

### ハイパーパラメータ

In [ ]:
from transformers import TrainingArguments, Trainer
from transformers import EarlyStoppingCallback

training_args = TrainingArguments(
    output_dir="test_trainer",
    eval_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=5,
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    warmup_steps=0.1,  # 最初の10%のステップで学習率を徐々に上げる
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    save_total_limit=1,
    seed=42,
    report_to="none",  # Weights & Biases などへのログ送信をしない
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

In [ ]:
trainer.train()

In [ ]:
# 2026年10月に試したときは、テストデータでの正解率・F1 はどちらも0.95〜0.97程度でした（GPU の計算には毎回わずかな揺らぎがあるので、結果は少し変わります）
trainer.evaluate(eval_dataset=test_dataset)

## ファインチューニングしたモデルで推論

ファインチューニングしたモデルを使って、テキストを分類してみます。

In [ ]:
classifier = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1,
)

In [ ]:
classifier(
    "私の隣室には日本人女性が住んでいますが、彼女も同じ日に発熱して病院に行きました。 彼女と私は一緒に治療を受けているため、お互いに接触している可能性が高いです。",
    truncation=True,
    max_length=512,
)

## テストデータの混同行列と誤り分析

`trainer.predict(test_dataset)` を使ってテストデータ全体の予測を取得し、混同行列（confusion matrix）と分類レポートを確認します。
さらに、誤分類された事例を元の「コメント」列と一緒に表示することで、`ch2_tfidf_baseline.ipynb` で扱った TF-IDF ベースラインと比べて BERT がどのような文脈（否定・仮定・他者の感染など）を正しく捉えられるようになったか、またどのような事例が依然として難しいかを分析できます。

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

pred_output = trainer.predict(test_dataset)
logits = pred_output.predictions
y_true = pred_output.label_ids
y_pred = np.argmax(logits, axis=-1)

# softmax で予測クラスの確信度を計算する
exp_logits = np.exp(logits - np.max(logits, axis=-1, keepdims=True))
probs = exp_logits / exp_logits.sum(axis=-1, keepdims=True)
confidences = probs[np.arange(len(y_pred)), y_pred]

label_names = ["FALSE", "TRUE"]
cm = pd.DataFrame(
    confusion_matrix(y_true, y_pred),
    index=[f"正解:{name}" for name in label_names],
    columns=[f"予測:{name}" for name in label_names],
)
print("混同行列:")
display(cm)
print("\n分類レポート:")
print(classification_report(y_true, y_pred, target_names=label_names, digits=4))

In [ ]:
test_raw = train_test_data["test"].to_pandas()
error_df = pd.DataFrame({
    "正解": [label_names[i] for i in y_true],
    "予測": [label_names[i] for i in y_pred],
    "確信度": np.round(confidences, 4),
    "文書": test_raw["text"],
    "コメント": test_raw["comment"],
})
misclassified = error_df[error_df["正解"] != error_df["予測"]].sort_values("確信度", ascending=False)
print(f"誤分類件数: {len(misclassified)} / {len(error_df)}")
misclassified